# CIFAR-10 Grad-CAM Research Project

**Goal:** Implement Grad-CAM from scratch, train three CIFAR-10 model variants, validate against `pytorch-grad-cam` (Spearman r > 0.95), and analyse heatmaps.

**Models:**
- `BaselineCNN` — 3-layer custom CNN from scratch
- `ResNet18-scratch` — ResNet18 with random init
- `ResNet18-pretrained` — ResNet18 fine-tuned from ImageNet weights

**Pipeline sections:**
1. Setup & Config
2. Utilities
3. Data Pipeline
4. Model Definitions
5. Training Loop
6. Grad-CAM Implementation
7. Evaluation
8. Adebayo Sanity Checks
9. Full Pipeline Orchestration

---
## 1. Setup & Dependencies
Install any missing packages and create the required output directories.

In [12]:
# Install required packages (uncomment if not already installed)
# !pip install torch torchvision numpy matplotlib scipy pyyaml grad-cam

import os, math, random, time, logging, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.models as tv_models
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use('Agg')          # headless rendering — no display required
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import spearmanr

print('All imports successful.')

All imports successful.


In [13]:
# Create output directory structure
for d in ['outputs/heatmaps', 'outputs/curves', 'outputs/sanity_checks', 'outputs/eval', 'models', 'data']:
    Path(d).mkdir(parents=True, exist_ok=True)
print('Output directories ready.')

Output directories ready.


---
## 2. Configuration
All hyper-parameters and paths in one place. Every value is explained.

In [14]:
# ─────────────────────────────────────────────
# Master config dict (mirrors config.yaml)
# ─────────────────────────────────────────────
CFG = {
    'seed': 42,            # Fixed for full reproducibility across runs
    'data': {
        'dataset':     'cifar10',
        'root':        './data',
        'num_workers': 0,
        # Normalization stats computed from TRAINING SET ONLY (prevents data leakage)
        'mean': [0.4914, 0.4822, 0.4465],
        'std':  [0.2470, 0.2435, 0.2616],
    },
    'augmentation': {
        'random_crop_size':    32,
        'random_crop_padding': 4,
        # Horizontal flip valid — objects appear mirrored in nature
        # NO vertical flip — objects don't appear upside-down in real photos
        'horizontal_flip_prob': 0.5,
    },
    'training': {
        'epochs':       100,
        'batch_size':   128,
        'lr':           0.01,   # Lower than typical 0.1 for pretrained variant
        'momentum':     0.9,
        'weight_decay': 5e-4,
    },
    'models': {
        'save_dir': './models',
        'baseline_cnn': {'dropout': 0.5},
    },
    'evaluation': {
        'library_parity_threshold': 0.95,   # Spearman r must exceed this to validate
    },
    'paths': {
        'outputs':       './outputs',
        'heatmaps':      './outputs/heatmaps',
        'curves':        './outputs/curves',
        'sanity_checks': './outputs/sanity_checks',
        'eval':          './outputs/eval',
    },
}

print('Config loaded.')

Config loaded.


---
## 3. Utilities
Reproducibility, logging, checkpoint I/O, and plotting helpers.

In [15]:
# ─────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────

def set_seed(seed: int = 42) -> None:
    """Fix all sources of randomness for full reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'[seed] All RNG sources fixed to {seed}')


# ─────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────

def get_logger(name: str, log_file: str = None) -> logging.Logger:
    """Returns a timestamped logger writing to stdout (and optionally a file)."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter('%(asctime)s | %(levelname)s | %(name)s | %(message)s',
                            datefmt='%Y-%m-%d %H:%M:%S')
    if not logger.handlers:
        ch = logging.StreamHandler()
        ch.setFormatter(fmt)
        logger.addHandler(ch)
        if log_file:
            Path(log_file).parent.mkdir(parents=True, exist_ok=True)
            fh = logging.FileHandler(log_file)
            fh.setFormatter(fmt)
            logger.addHandler(fh)
    return logger


# ─────────────────────────────────────────────
# Checkpoint I/O
# ─────────────────────────────────────────────

def save_checkpoint(state: dict, path: str) -> None:
    """Save model + optimizer state + metrics to a .pth file."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)
    print(f'[checkpoint] Saved → {path}')


def load_checkpoint(path: str, model: nn.Module, optimizer=None, device='cpu'):
    """Load checkpoint into model (and optionally optimizer). Returns full state dict."""
    state = torch.load(path, map_location=device)
    model.load_state_dict(state['model_state_dict'])
    if optimizer and 'optimizer_state_dict' in state:
        optimizer.load_state_dict(state['optimizer_state_dict'])
    print(f"[checkpoint] Loaded ← {path}  (epoch {state.get('epoch', '?')})")
    return state


# ─────────────────────────────────────────────
# Image helper
# ─────────────────────────────────────────────

def denormalize(tensor: torch.Tensor, mean: list, std: list) -> np.ndarray:
    """
    Undo normalization for visualization.
    tensor: CHW float tensor → returns HWC uint8 numpy array.
    """
    m = torch.tensor(mean).view(3, 1, 1)
    s = torch.tensor(std).view(3, 1, 1)
    img = tensor.cpu().float() * s + m
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()
    return (img * 255).astype(np.uint8)


# ─────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────

def plot_training_curves(train_losses, val_losses, train_accs, val_accs,
                         save_path: str, title: str = 'Training Curves') -> None:
    """Save loss + accuracy curves side-by-side. Monitor the gen gap (val_loss - train_loss)."""
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    epochs = range(1, len(train_losses) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    axes[0].plot(epochs, train_losses, label='Train', color='#2563EB')
    axes[0].plot(epochs, val_losses,   label='Val',   color='#DC2626', linestyle='--')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, train_accs, label='Train', color='#2563EB')
    axes[1].plot(epochs, val_accs,   label='Val',   color='#DC2626', linestyle='--')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[plot] Training curves saved → {save_path}')


# ─────────────────────────────────────────────
# Device
# ─────────────────────────────────────────────

def get_device() -> torch.device:
    """Return CUDA if available, else CPU."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[device] Using: {device}'
          + (f' ({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else ''))
    return device


# Initialise seed and device now
set_seed(CFG['seed'])
DEVICE = get_device()

[seed] All RNG sources fixed to 42
[device] Using: cuda (NVIDIA RTX 5000 Ada Generation)


---
## 4. Data Pipeline
CIFAR-10 loading, augmentation, class-balance verification, and sample grid.

In [16]:
# CIFAR-10 class names (index-aligned)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']


def get_transforms(cfg: dict, split: str):
    """
    Train: RandomCrop + HorizontalFlip + Normalize
    Val/Test: Normalize only (deterministic evaluation)

    No vertical flip — objects don't appear upside-down in natural images.
    """
    mean = cfg['data']['mean']
    std  = cfg['data']['std']
    aug  = cfg['augmentation']
    normalize = T.Normalize(mean=mean, std=std)

    if split == 'train':
        return T.Compose([
            T.RandomCrop(aug['random_crop_size'], padding=aug['random_crop_padding']),
            T.RandomHorizontalFlip(p=aug['horizontal_flip_prob']),
            T.ToTensor(),
            normalize,
        ])
    else:
        return T.Compose([T.ToTensor(), normalize])


def get_dataloaders(cfg: dict):
    """
    Build train / val / test DataLoaders.
    CIFAR-10: 50k train + 10k test. Full test set used as val (academic convention).
    Returns: train_loader, val_loader, test_loader
    """
    root = cfg['data']['root']
    bs   = cfg['training']['batch_size']
    nw   = cfg['data']['num_workers']

    train_set = torchvision.datasets.CIFAR10(
        root=root, train=True,  download=True, transform=get_transforms(cfg, 'train'))
    test_set  = torchvision.datasets.CIFAR10(
        root=root, train=False, download=True, transform=get_transforms(cfg, 'val'))

    train_loader = DataLoader(train_set, batch_size=bs, shuffle=True,
                              num_workers=nw, pin_memory=True)
    val_loader   = DataLoader(test_set,  batch_size=bs, shuffle=False,
                              num_workers=nw, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=bs, shuffle=False,
                              num_workers=nw, pin_memory=True)

    print(f'Train: {len(train_set):,} samples | Val/Test: {len(test_set):,} samples')
    return train_loader, val_loader, test_loader


def verify_class_balance(dataset, split_name: str) -> None:
    """
    CIFAR-10 is perfectly balanced: 5000 train / 1000 test per class.
    Assertion failure means something is wrong with dataset loading.
    """
    labels  = [dataset[i][1] for i in range(len(dataset))]
    counts  = np.bincount(labels, minlength=10)
    expected = 5000 if split_name == 'train' else 1000
    print(f'\n[{split_name}] Class counts:')
    for i, (cls, cnt) in enumerate(zip(CLASSES, counts)):
        print(f'  {i:2d}. {cls:<12s}: {cnt}')
    for i, cnt in enumerate(counts):
        assert cnt == expected, f'Class {CLASSES[i]} has {cnt} samples, expected {expected}.'
    print(f'[{split_name}] ✓ Class balance verified ({expected} per class)')


def save_sample_grid(dataset, cfg: dict, save_path: str, n: int = 16) -> None:
    """Save grid of n augmented training images — visual label sanity check."""
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean']
    std  = cfg['data']['std']
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle('Augmented training samples (sanity check)', fontsize=11)
    for ax, idx in zip(axes.flat, indices):
        img_tensor, label = dataset[int(idx)]
        ax.imshow(denormalize(img_tensor, mean, std))
        ax.set_title(CLASSES[label], fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[sanity] Sample grid saved → {save_path}')


def run_data_sanity_checks(cfg: dict) -> None:
    """Run all data-level sanity checks. Call once before training."""
    root = cfg['data']['root']
    raw_train = torchvision.datasets.CIFAR10(root=root, train=True,  download=True, transform=T.ToTensor())
    raw_test  = torchvision.datasets.CIFAR10(root=root, train=False, download=True, transform=T.ToTensor())
    verify_class_balance(raw_train, 'train')
    verify_class_balance(raw_test,  'test')
    aug_train = torchvision.datasets.CIFAR10(
        root=root, train=True, download=True, transform=get_transforms(cfg, 'train'))
    save_sample_grid(aug_train, cfg,
                     save_path=cfg['paths']['sanity_checks'] + '/data_samples.png')
    print('[data] All data sanity checks passed ✓')

In [17]:
# ── DAY 2: Run data sanity checks ────────────────────────────────
# Expected output:
#   [train] ✓ Class balance verified (5000 per class)
#   [test]  ✓ Class balance verified (1000 per class)
#   Sample grid saved → outputs/sanity_checks/data_samples.png
run_data_sanity_checks(CFG)


[train] Class counts:
   0. airplane    : 5000
   1. automobile  : 5000
   2. bird        : 5000
   3. cat         : 5000
   4. deer        : 5000
   5. dog         : 5000
   6. frog        : 5000
   7. horse       : 5000
   8. ship        : 5000
   9. truck       : 5000
[train] ✓ Class balance verified (5000 per class)

[test] Class counts:
   0. airplane    : 1000
   1. automobile  : 1000
   2. bird        : 1000
   3. cat         : 1000
   4. deer        : 1000
   5. dog         : 1000
   6. frog        : 1000
   7. horse       : 1000
   8. ship        : 1000
   9. truck       : 1000
[test] ✓ Class balance verified (1000 per class)
[sanity] Sample grid saved → ./outputs/sanity_checks/data_samples.png
[data] All data sanity checks passed ✓


---
## 5. Model Definitions

Three models with distinct scientific motivations:
- **BaselineCNN**: 3-conv scratch model — simple comparison target, coarse Grad-CAM maps
- **ResNet18-scratch**: shows what architecture depth alone buys
- **ResNet18-pretrained**: shows transfer learning benefit; sharpest Grad-CAM maps expected

In [18]:
# ─────────────────────────────────────────────
# 1. Baseline CNN
# ─────────────────────────────────────────────

class BaselineCNN(nn.Module):
    """
    3-layer CNN. Input: 3×32×32 → Output: 10 logits.
    Spatial resolution: 32 → 16 → 8 → 4 (via MaxPool after each block).
    layer3 is the Grad-CAM target (final conv layer, 4×4 feature maps).
    """
    def __init__(self, num_classes: int = 10, dropout: float = 0.5):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 32→16

        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2))  # 16→8

        # Grad-CAM hooks on layer3
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2)) # 8→4

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.layer3(self.layer2(self.layer1(x))))


# ─────────────────────────────────────────────
# 2. ResNet18 (pretrained or scratch)
# ─────────────────────────────────────────────

def build_resnet18(pretrained: bool = True, num_classes: int = 10) -> nn.Module:
    """
    ResNet18 adapted for 32×32 CIFAR-10 input:
      - First conv: 7×7 stride-2 → 3×3 stride-1  (avoids shrinking tiny spatial dims)
      - Initial MaxPool replaced with Identity     (same reason)
      - Final FC: 1000 → num_classes
    """
    if pretrained:
        weights = tv_models.ResNet18_Weights.IMAGENET1K_V1
        model   = tv_models.resnet18(weights=weights)
        print('[model] ResNet18 loaded with ImageNet pretrained weights')
    else:
        model = tv_models.resnet18(weights=None)
        print('[model] ResNet18 initialized from scratch (random weights)')

    # Adapt for 32×32 input
    model.conv1  = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()   # Remove initial MaxPool

    # Replace classifier head
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


# ─────────────────────────────────────────────
# Model factory
# ─────────────────────────────────────────────

def build_model(model_name: str, cfg: dict) -> nn.Module:
    """Factory function. Options: 'baseline_cnn', 'resnet18_pretrained', 'resnet18_scratch'."""
    if model_name == 'baseline_cnn':
        return BaselineCNN(num_classes=10, dropout=cfg['models']['baseline_cnn']['dropout'])
    elif model_name == 'resnet18_pretrained':
        return build_resnet18(pretrained=True,  num_classes=10)
    elif model_name == 'resnet18_scratch':
        return build_resnet18(pretrained=False, num_classes=10)
    else:
        raise ValueError(f'Unknown model: {model_name}')


def count_parameters(model: nn.Module) -> int:
    """Return number of trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick parameter count check
for name in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    m = build_model(name, CFG)
    print(f'{name}: {count_parameters(m):,} trainable parameters')

baseline_cnn: 620,810 trainable parameters
[model] ResNet18 initialized from scratch (random weights)
resnet18_scratch: 11,173,962 trainable parameters
[model] ResNet18 loaded with ImageNet pretrained weights
resnet18_pretrained: 11,173,962 trainable parameters


---
## 6. Training Loop

Key scientific practices:
1. **Epoch-1 loss check**: expected ≈ 2.3026 = ln(10) for random 10-class output
2. **Generalization gap monitoring**: `val_loss - train_loss` at every epoch
3. **Best checkpoint** saved by val accuracy; final checkpoint saved separately
4. **Training curves** saved after each run

In [19]:
train_logger = get_logger('train', log_file='outputs/train.log')


def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    """Single training epoch. Returns (avg_loss, accuracy%)."""
    model.train()
    total_loss = total_correct = total_samples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Single evaluation pass. Returns (avg_loss, accuracy%)."""
    model.eval()
    total_loss = total_correct = total_samples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss   = criterion(logits, labels)
        total_loss    += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += images.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples


def check_initial_loss(loss: float, num_classes: int = 10, tol: float = 0.3) -> None:
    """
    Epoch-1 sanity check. For a randomly initialized model on balanced data:
      expected CE loss = -ln(1/10) = ln(10) ≈ 2.3026
    Loss >> 2.3 → weight init problem or NaN in pipeline
    Loss << 2.3 → data leakage or model loaded with trained weights
    """
    expected  = math.log(num_classes)
    deviation = abs(loss - expected)
    status    = '✓ PASSED' if deviation <= tol else '✗ FAILED'
    train_logger.info(
        f'[sanity] Epoch-1 loss: got {loss:.4f}, expected ~{expected:.4f} '
        f'(±{tol}) | {status}'
    )
    if deviation > tol:
        train_logger.warning(
            '[sanity] Check: weight init, data pipeline, class balance, label encoding.')


def train(model_name: str, cfg: dict, epochs_override: int = None) -> nn.Module:
    """
    Full training run for one model variant.
    Saves: best checkpoint, final checkpoint, training curves.
    Returns the best model loaded from checkpoint.
    """
    set_seed(cfg['seed'])
    device = get_device()

    train_loader, val_loader, _ = get_dataloaders(cfg)
    model     = build_model(model_name, cfg).to(device)
    n_params  = count_parameters(model)
    train_logger.info(f'[model] {model_name} | {n_params:,} trainable params')

    # SGD + momentum: still the standard for CIFAR-10 (Adam overfits more here)
    optimizer = optim.SGD(
        model.parameters(),
        lr=cfg['training']['lr'],
        momentum=cfg['training']['momentum'],
        weight_decay=cfg['training']['weight_decay'],
    )
    num_epochs = epochs_override or cfg['training']['epochs']
    # Cosine annealing: smooth LR decay, avoids abrupt drops
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    criterion  = nn.CrossEntropyLoss()

    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    best_val_acc = 0.0
    save_dir     = cfg['models']['save_dir']
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    train_logger.info(f'Starting: {model_name} | {num_epochs} epochs')

    for epoch in range(1, num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch)
        val_loss,   val_acc   = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        if epoch == 1:
            check_initial_loss(train_loss)

        train_losses.append(train_loss);  val_losses.append(val_loss)
        train_accs.append(train_acc);     val_accs.append(val_acc)

        gen_gap = val_loss - train_loss   # positive = overfitting
        train_logger.info(
            f'Epoch {epoch:3d}/{num_epochs} | '
            f'TrainLoss={train_loss:.4f} TrainAcc={train_acc:.1f}% | '
            f'ValLoss={val_loss:.4f} ValAcc={val_acc:.1f}% | '
            f'Gap={gen_gap:+.4f} | LR={scheduler.get_last_lr()[0]:.6f} | '
            f'{time.time()-t0:.1f}s'
        )

        # Save best checkpoint (by val accuracy)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint({
                'epoch': epoch, 'model_name': model_name,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc, 'val_loss': val_loss, 'train_loss': train_loss,
            }, path=f'{save_dir}/{model_name}_best.pth')

    # Save final checkpoint and curves
    save_checkpoint({
        'epoch': num_epochs, 'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'train_losses': train_losses, 'val_losses': val_losses,
        'train_accs': train_accs,     'val_accs': val_accs,
    }, path=f'{save_dir}/{model_name}_final.pth')

    plot_training_curves(
        train_losses, val_losses, train_accs, val_accs,
        save_path=f"{cfg['paths']['curves']}/{model_name}_curves.png",
        title=f'{model_name} — Training Curves (best val: {best_val_acc:.1f}%)'
    )
    train_logger.info(f'Training complete. Best val accuracy: {best_val_acc:.2f}%')

    # Load and return best model for downstream use
    best_model = build_model(model_name, cfg).to(device)
    load_checkpoint(f'{save_dir}/{model_name}_best.pth', best_model, device=str(device))
    return best_model

In [20]:
# ── DAY 3–5: Train all three models ──────────────────────────────
# Tip: set epochs_override=5 for a quick smoke-test before full runs
# GPU recommended (~5 min/model); CPU works but is significantly slower

TRAINED_MODELS = {}

for model_name in ['baseline_cnn', 'resnet18_scratch', 'resnet18_pretrained']:
    print('\n' + '='*60)
    print(f'Training: {model_name}')
    print('='*60)
    TRAINED_MODELS[model_name] = train(model_name, CFG)   # remove epochs_override for full run
    # TRAINED_MODELS[model_name] = train(model_name, CFG, epochs_override=5)  # quick test

print('\nAll models trained.')


Training: baseline_cnn
[seed] All RNG sources fixed to 42
[device] Using: cuda (NVIDIA RTX 5000 Ada Generation)


2026-05-15 07:15:55 | INFO | train | [model] baseline_cnn | 620,810 trainable params
2026-05-15 07:15:55 | INFO | train | Starting: baseline_cnn | 100 epochs


Train: 50,000 samples | Val/Test: 10,000 samples


2026-05-15 07:16:31 | INFO | train | [sanity] Epoch-1 loss: got 1.5917, expected ~2.3026 (±0.3) | ✗ FAILED
2026-05-15 07:16:31 | WARNING | train | [sanity] Check: weight init, data pipeline, class balance, label encoding.
2026-05-15 07:16:31 | INFO | train | Epoch   1/100 | TrainLoss=1.5917 TrainAcc=41.1% | ValLoss=1.1987 ValAcc=55.9% | Gap=-0.3930 | LR=0.009998 | 36.2s


[checkpoint] Saved → ./models/baseline_cnn_best.pth


...

[checkpoint] Saved → ./models/resnet18_pretrained_best.pth


2026-05-15 10:22:30 | INFO | train | Epoch  71/100 | TrainLoss=0.0011 TrainAcc=100.0% | ValLoss=0.1664 ValAcc=95.7% | Gap=+0.1652 | LR=0.001935 | 45.7s
2026-05-15 10:23:15 | INFO | train | Epoch  72/100 | TrainLoss=0.0013 TrainAcc=100.0% | ValLoss=0.1686 ValAcc=95.6% | Gap=+0.1673 | LR=0.001813 | 45.6s
2026-05-15 10:24:01 | INFO | train | Epoch  73/100 | TrainLoss=0.0011 TrainAcc=100.0% | ValLoss=0.1657 ValAcc=95.7% | Gap=+0.1646 | LR=0.001693 | 45.7s
2026-05-15 10:24:47 | INFO | train | Epoch  74/100 | TrainLoss=0.0011 TrainAcc=100.0% | ValLoss=0.1634 ValAcc=95.7% | Gap=+0.1623 | LR=0.001577 | 46.1s
2026-05-15 10:25:45 | INFO | train | Epoch  75/100 | TrainLoss=0.0013 TrainAcc=100.0% | ValLoss=0.1680 ValAcc=95.6% | Gap=+0.1667 | LR=0.001464 | 45.6s
2026-05-15 10:26:31 | INFO | train | Epoch  76/100 | TrainLoss=0.0011 TrainAcc=100.0% | ValLoss=0.1669 ValAcc=95.7% | Gap=+0.1657 | LR=0.001355 | 45.9s
2026-05-15 10:27:17 | INFO | train | Epoch  77/100 | TrainLoss=0.0010 TrainAcc=100.0% | 

[checkpoint] Saved → ./models/resnet18_pretrained_final.pth


2026-05-15 10:42:41 | INFO | train | Training complete. Best val accuracy: 95.86%


[plot] Training curves saved → ./outputs/curves/resnet18_pretrained_curves.png
[model] ResNet18 loaded with ImageNet pretrained weights
[checkpoint] Loaded ← ./models/resnet18_pretrained_best.pth  (epoch 70)

All models trained.


---
## 7. Grad-CAM Implementation (from scratch)

**Mathematical pipeline:**
1. Forward pass → save feature maps at target layer via hook
2. Compute raw logit for target class (**not** softmax — avoids class competition)
3. Backward pass → save gradients via hook
4. Importance weights: `mean(gradients, dims=[H,W])` — one scalar per channel
5. Weighted sum of feature map channels
6. ReLU — keep only positively contributing features
7. Upsample to input resolution (32×32)
8. Normalize to [0, 1]

In [21]:
class GradCAM:
    """
    Grad-CAM from scratch using PyTorch forward and backward hooks.

    Why raw logit (not softmax)?
        Softmax introduces inter-class competition: a high score for class A
        suppresses class B even if strong features are present for B. Raw
        logits isolate one class without this suppression.

    Why ReLU at the end?
        Some channels negatively contribute (suppress) the target class.
        ReLU zeroes those out — we only care about pixels that HELP identify
        the target class.
    """

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model
        self.target_layer = target_layer
        self._feature_maps = None
        self._gradients    = None
        self._register_hooks()

    def _register_hooks(self) -> None:
        """Attach forward and backward hooks to the target layer."""
        def forward_hook(module, inp, out):
            self._feature_maps = out.detach()   # (B, C, H, W)

        def backward_hook(module, grad_in, grad_out):
            self._gradients = grad_out[0].detach()  # (B, C, H, W)

        self._fwd = self.target_layer.register_forward_hook(forward_hook)
        self._bwd = self.target_layer.register_full_backward_hook(backward_hook)

    def remove_hooks(self) -> None:
        """Always call when done to prevent memory leaks."""
        self._fwd.remove()
        self._bwd.remove()

    def __call__(self, input_tensor: torch.Tensor,
                 target_class: int = None,
                 input_size: tuple = (32, 32)) -> np.ndarray:
        """
        Compute Grad-CAM heatmap for a single image.
        input_tensor: (1, C, H, W) normalized tensor
        Returns: (H, W) float array in [0, 1]
        """
        self.model.eval()
        self.model.zero_grad()

        logits = self.model(input_tensor)                        # Step 1: forward

        if target_class is None:
            target_class = logits.argmax(dim=1).item()          # default: predicted class

        score = logits[0, target_class]                          # Step 2-3: raw logit
        score.backward()                                         # Step 4: backward

        weights = self._gradients.mean(dim=(2, 3))               # Step 5: (1, C)
        cam = (weights[:, :, None, None] * self._feature_maps).sum(dim=1).squeeze(0)  # Step 6
        cam = F.relu(cam)                                        # Step 7: ReLU

        cam = F.interpolate(                                     # Step 8: upsample
            cam.unsqueeze(0).unsqueeze(0),
            size=input_size, mode='bilinear', align_corners=False
        ).squeeze().cpu().numpy()

        # Step 9: normalize to [0, 1]
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
        return cam

    def __enter__(self): return self
    def __exit__(self, *args): self.remove_hooks()


def get_target_layer(model: nn.Module, model_name: str) -> nn.Module:
    """
    Return the correct Grad-CAM target layer for each model.
      ResNet18:    model.layer4[-1]  — final residual block (highest semantics)
      BaselineCNN: model.layer3      — final conv block
    """
    if 'resnet18' in model_name:
        return model.layer4[-1]
    elif 'baseline_cnn' in model_name:
        return model.layer3
    else:
        raise ValueError(f'Unknown target layer for: {model_name}')


def overlay_heatmap(image_np: np.ndarray, cam: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Blend jet-colored CAM heatmap onto original image. Returns HWC uint8."""
    heatmap = (cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)
    blended = (1 - alpha) * image_np.astype(np.float32) / 255 + \
               alpha      * heatmap.astype(np.float32) / 255
    return (blended * 255).astype(np.uint8)


def save_heatmap_grid(model, model_name, dataset, cfg, save_path,
                      device=torch.device('cpu')) -> None:
    """
    Save 10×2 grid of [original | Grad-CAM overlay] for all CIFAR-10 classes.
    One example image per class.
    """
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean'];  std = cfg['data']['std']
    target_layer = get_target_layer(model, model_name)

    # Collect one image per class
    class_images = {}
    for idx in range(len(dataset)):
        img_t, label = dataset[idx]
        if label not in class_images:
            class_images[label] = (img_t, label)
        if len(class_images) == 10:
            break

    fig, axes = plt.subplots(10, 2, figsize=(5, 25))
    fig.suptitle(f'Grad-CAM — {model_name}', fontsize=11, fontweight='bold')

    with GradCAM(model, target_layer) as gcam:
        for cls_idx in range(10):
            img_t, _ = class_images[cls_idx]
            img_np   = denormalize(img_t, mean, std)
            cam      = gcam(img_t.unsqueeze(0).to(device), target_class=cls_idx)
            overlay  = overlay_heatmap(img_np, cam)
            axes[cls_idx, 0].imshow(img_np)
            axes[cls_idx, 0].set_ylabel(CLASSES[cls_idx], fontsize=7,
                                        rotation=0, labelpad=40, va='center')
            axes[cls_idx, 0].axis('off')
            axes[cls_idx, 1].imshow(overlay)
            axes[cls_idx, 1].axis('off')

    axes[0, 0].set_title('Original', fontsize=9)
    axes[0, 1].set_title('Grad-CAM', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[gradcam] Heatmap grid saved → {save_path}')

In [22]:
# ── DAY 9: Generate Grad-CAM heatmap grids for all 3 models ─────
# Expected: pretrained shows sharpest, most semantically localised heatmaps

test_dataset = torchvision.datasets.CIFAR10(
    root=CFG['data']['root'], train=False, download=True,
    transform=get_transforms(CFG, 'val')
)

for model_name, model in TRAINED_MODELS.items():
    model.eval()
    save_heatmap_grid(
        model, model_name, test_dataset, CFG,
        save_path=f"{CFG['paths']['heatmaps']}/{model_name}_gradcam_grid.png",
        device=DEVICE
    )

print('\nHeatmap grids saved to outputs/heatmaps/')

[gradcam] Heatmap grid saved → ./outputs/heatmaps/baseline_cnn_gradcam_grid.png
[gradcam] Heatmap grid saved → ./outputs/heatmaps/resnet18_scratch_gradcam_grid.png
[gradcam] Heatmap grid saved → ./outputs/heatmaps/resnet18_pretrained_gradcam_grid.png

Heatmap grids saved to outputs/heatmaps/


---
## 8. Library Parity Validation
Compare our Grad-CAM against `pytorch-grad-cam` using **Spearman rank correlation**.
Threshold: **r > 0.95** — as specified in `config.yaml`.

In [23]:
def verify_against_library(model, model_name: str, image_tensor: torch.Tensor,
                            target_class: int, cfg: dict,
                            threshold: float = 0.95,
                            device=torch.device('cpu')) -> float:
    """
    Compute Spearman rank correlation between our Grad-CAM and pytorch-grad-cam.
    r > threshold confirms our implementation is correct.
    This is the methodological validation step required for the report.
    """
    try:
        from pytorch_grad_cam import GradCAM as LibGradCAM
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    except ImportError:
        print('[parity] pytorch-grad-cam not installed. Run: pip install grad-cam')
        return float('nan')

    target_layer = get_target_layer(model, model_name)
    input_t      = image_tensor.unsqueeze(0).to(device)

    # Our implementation
    with GradCAM(model, target_layer) as gcam:
        our_cam = gcam(input_t, target_class=target_class)

    # Reference library implementation
    lib_gcam = LibGradCAM(model=model, target_layers=[target_layer])
    lib_cam  = lib_gcam(input_tensor=input_t,
                        targets=[ClassifierOutputTarget(target_class)])[0]

    r, p = spearmanr(our_cam.flatten(), lib_cam.flatten())
    status = '✓ PASSED' if r >= threshold else '✗ FAILED'
    print(f'[parity] {model_name}: Spearman r = {r:.4f} (p={p:.2e}) '
          f'| threshold = {threshold} | {status}')
    if r < threshold:
        print('[parity] Debug: check target layer, backward hook type, averaging dims.')
    return r


# ── DAY 10: Run parity check on one test image per model ─────────
# Install the reference library first if needed:
# !pip install grad-cam

parity_results = {}
sample_img, sample_label = test_dataset[0]
threshold = CFG['evaluation']['library_parity_threshold']

for model_name, model in TRAINED_MODELS.items():
    model.eval()
    r = verify_against_library(
        model, model_name, sample_img, sample_label, CFG,
        threshold=threshold, device=DEVICE
    )
    parity_results[model_name] = r

print('\nParity Results Summary:')
for name, r in parity_results.items():
    status = '✓' if (not np.isnan(r) and r >= threshold) else '✗'
    print(f'  {status} {name}: r = {r:.4f}')

[parity] baseline_cnn: Spearman r = 1.0000 (p=0.00e+00) | threshold = 0.95 | ✓ PASSED
[parity] resnet18_scratch: Spearman r = 1.0000 (p=0.00e+00) | threshold = 0.95 | ✓ PASSED
[parity] resnet18_pretrained: Spearman r = 1.0000 (p=0.00e+00) | threshold = 0.95 | ✓ PASSED

Parity Results Summary:
  ✓ baseline_cnn: r = 1.0000
  ✓ resnet18_scratch: r = 1.0000
  ✓ resnet18_pretrained: r = 1.0000


---
## 9. Evaluation — Accuracy & Confusion Matrices
Full test-set evaluation + per-model confusion matrices.

In [24]:
def run_full_evaluation(trained_models: dict, cfg: dict, device) -> dict:
    """
    Evaluate all models on the full test set.
    Saves: per-model confusion matrices + accuracy bar chart.
    Returns: dict of {model_name: {'accuracy': ..., 'loss': ...}}
    """
    _, val_loader, _ = get_dataloaders(cfg)
    criterion        = nn.CrossEntropyLoss()
    eval_path        = cfg['paths']['eval']
    Path(eval_path).mkdir(parents=True, exist_ok=True)

    results = {}

    for model_name, model in trained_models.items():
        model.eval()

        # Collect all predictions
        all_preds, all_labels = [], []
        total_loss = total_samples = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits  = model(images)
                loss    = criterion(logits, labels)
                total_loss    += loss.item() * images.size(0)
                total_samples += images.size(0)
                all_preds.extend(logits.argmax(1).cpu().tolist())
                all_labels.extend(labels.cpu().tolist())

        acc  = 100.0 * sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
        loss = total_loss / total_samples
        results[model_name] = {'accuracy': acc, 'loss': loss}
        print(f'[eval] {model_name}: Acc={acc:.2f}%  Loss={loss:.4f}')

        # Build confusion matrix
        cm_matrix = np.zeros((10, 10), dtype=int)
        for pred, true in zip(all_preds, all_labels):
            cm_matrix[true, pred] += 1

        # Plot confusion matrix
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm_matrix, interpolation='nearest', cmap='Blues')
        plt.colorbar(im)
        ax.set(xticks=range(10), yticks=range(10),
               xticklabels=CLASSES, yticklabels=CLASSES,
               xlabel='Predicted', ylabel='True',
               title=f'{model_name} — Confusion Matrix (Acc={acc:.1f}%)')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        thresh = cm_matrix.max() / 2
        for i in range(10):
            for j in range(10):
                ax.text(j, i, cm_matrix[i, j], ha='center', va='center',
                        color='white' if cm_matrix[i, j] > thresh else 'black', fontsize=7)
        plt.tight_layout()
        save_p = f'{eval_path}/{model_name}_confusion.png'
        plt.savefig(save_p, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'[eval] Confusion matrix saved → {save_p}')

    # Accuracy comparison bar chart
    names = list(results.keys())
    accs  = [results[n]['accuracy'] for n in names]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(names, accs, color=['#3B82F6', '#10B981', '#F59E0B'], width=0.5)
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Model Accuracy Comparison — CIFAR-10')
    ax.set_ylim(0, 105)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{eval_path}/accuracy_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[eval] Accuracy comparison saved → {eval_path}/accuracy_comparison.png")

    return results


# ── DAY 8: Run evaluation ────────────────────────────────────────
eval_results = run_full_evaluation(TRAINED_MODELS, CFG, DEVICE)

print('\nFinal Accuracy Table:')
print(f'{"Model":<25} {"Accuracy":>10} {"Loss":>10}')
print('-' * 47)
for name, res in eval_results.items():
    print(f'{name:<25} {res["accuracy"]:>9.2f}% {res["loss"]:>10.4f}')

Train: 50,000 samples | Val/Test: 10,000 samples
[eval] baseline_cnn: Acc=87.28%  Loss=0.3825
[eval] Confusion matrix saved → ./outputs/eval/baseline_cnn_confusion.png
[eval] resnet18_scratch: Acc=93.35%  Loss=0.2823
[eval] Confusion matrix saved → ./outputs/eval/resnet18_scratch_confusion.png
[eval] resnet18_pretrained: Acc=95.86%  Loss=0.1640
[eval] Confusion matrix saved → ./outputs/eval/resnet18_pretrained_confusion.png
[eval] Accuracy comparison saved → ./outputs/eval/accuracy_comparison.png

Final Accuracy Table:
Model                       Accuracy       Loss
-----------------------------------------------
baseline_cnn                  87.28%     0.3825
resnet18_scratch              93.35%     0.2823
resnet18_pretrained           95.86%     0.1640


---
## 10. Adebayo Sanity Checks

**Why?** Adebayo et al. (2018) showed many saliency methods produce plausible-looking maps even on randomly initialised models — meaning they may reflect data statistics rather than model internals.

We run two tests:
1. **Model randomization**: Progressively randomise weights layer-by-layer (top→down). Maps should change visibly.
2. **Data randomization**: Re-train on shuffled labels. Maps should look different from those of the correctly trained model.

In [25]:
def model_randomization_test(model: nn.Module, model_name: str, image_tensor: torch.Tensor,
                              cfg: dict, target_class: int,
                              save_path: str, device=torch.device('cpu')) -> None:
    """
    Model Randomization Test (Adebayo et al.):
    Cascade-randomize weights from the top layer downward.
    At each stage, compute a Grad-CAM map and compare to the original.
    If maps change as weights are randomized → Grad-CAM is sensitive to model weights ✓
    If maps stay similar → saliency is not reflecting learned features ✗
    """
    import copy
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean'];  std = cfg['data']['std']
    input_t = image_tensor.unsqueeze(0).to(device)

    # Collect all named parameter layers (leaf modules with parameters)
    layers = [(name, mod) for name, mod in model.named_modules()
              if len(list(mod.parameters(recurse=False))) > 0]
    layers_to_randomize = layers[::-1]   # top → bottom
    stages = min(5, len(layers_to_randomize))  # show up to 5 stages

    model_copy = copy.deepcopy(model)
    target_layer = get_target_layer(model_copy, model_name)

    fig, axes = plt.subplots(1, stages + 1, figsize=((stages + 1) * 3, 3))
    fig.suptitle(f'Model Randomization Test — {model_name}', fontsize=10)

    # Stage 0: original model
    with GradCAM(model_copy, target_layer) as gcam:
        cam = gcam(input_t, target_class=target_class)
    img_np = denormalize(image_tensor, mean, std)
    axes[0].imshow(overlay_heatmap(img_np, cam))
    axes[0].set_title('Original', fontsize=8); axes[0].axis('off')

    # Progressive randomization
    step = max(1, len(layers_to_randomize) // stages)
    for i in range(1, stages + 1):
        # Randomize the next batch of layers
        for _, mod in layers_to_randomize[(i-1)*step : i*step]:
            for p in mod.parameters(recurse=False):
                nn.init.normal_(p)
        target_layer = get_target_layer(model_copy, model_name)
        with GradCAM(model_copy, target_layer) as gcam:
            cam = gcam(input_t, target_class=target_class)
        axes[i].imshow(overlay_heatmap(img_np, cam))
        axes[i].set_title(f'Rand stage {i}', fontsize=8)
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[sanity] Model randomization grid saved → {save_path}')


def data_randomization_test(model_name: str, cfg: dict,
                             test_dataset, target_class: int,
                             save_path: str, shuffle_epochs: int = 10,
                             device=torch.device('cpu')) -> None:
    """
    Data Randomization Test (Adebayo et al.):
    Train a model on SHUFFLED labels, then compare its Grad-CAM maps
    to those of the correctly trained model.
    If maps look similar → saliency reflects data statistics, not model internals.
    If maps differ noticeably → saliency is model-dependent ✓
    """
    from torch.utils.data import TensorDataset
    import copy

    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    mean = cfg['data']['mean'];  std = cfg['data']['std']

    print(f'[sanity] Training model on shuffled labels ({shuffle_epochs} epochs)...')

    # Load training data and shuffle labels
    root = cfg['data']['root']
    raw_train = torchvision.datasets.CIFAR10(
        root=root, train=True, download=True, transform=get_transforms(cfg, 'train'))
    shuffled_labels = torch.randperm(len(raw_train)) % 10   # random labels

    # Build a wrapper dataset with shuffled labels
    class ShuffledDataset(torch.utils.data.Dataset):
        def __init__(self, base, new_labels):
            self.base = base
            self.labels = new_labels
        def __len__(self): return len(self.base)
        def __getitem__(self, idx):
            img, _ = self.base[idx]
            return img, self.labels[idx].item()

    shuffled_ds     = ShuffledDataset(raw_train, shuffled_labels)
    shuffled_loader = DataLoader(shuffled_ds, batch_size=cfg['training']['batch_size'],
                                 shuffle=True, num_workers=cfg['data']['num_workers'])

    rand_model = build_model(model_name, cfg).to(device)
    optimizer  = optim.SGD(rand_model.parameters(), lr=cfg['training']['lr'],
                           momentum=cfg['training']['momentum'],
                           weight_decay=cfg['training']['weight_decay'])
    criterion  = nn.CrossEntropyLoss()

    for ep in range(1, shuffle_epochs + 1):
        loss, acc = train_one_epoch(rand_model, shuffled_loader, criterion, optimizer, device, ep)
        print(f'  Shuffle epoch {ep}/{shuffle_epochs}: loss={loss:.4f} acc={acc:.1f}%')

    # Compare heatmaps side-by-side
    img_t, _    = test_dataset[0]
    img_np      = denormalize(img_t, mean, std)
    input_t     = img_t.unsqueeze(0).to(device)

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    fig.suptitle(f'Data Randomization Test — {model_name}', fontsize=10)

    orig_model  = TRAINED_MODELS[model_name]
    for ax, (mdl, lbl) in zip(axes, [(orig_model, 'Trained (correct labels)'),
                                      (rand_model, 'Trained (shuffled labels)')]):
        tl = get_target_layer(mdl, model_name)
        with GradCAM(mdl, tl) as gcam:
            cam = gcam(input_t, target_class=target_class)
        ax.imshow(overlay_heatmap(img_np, cam))
        ax.set_title(lbl, fontsize=7)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'[sanity] Data randomization comparison saved → {save_path}')


# ── DAY 11: Run Adebayo sanity checks ────────────────────────────
# Note: data randomization re-trains a model — set shuffle_epochs=10 for a quick version

sanity_path = CFG['paths']['sanity_checks']
target_class = 0   # 'airplane' — change to test other classes

sample_img_t, _ = test_dataset[0]

for model_name, model in TRAINED_MODELS.items():
    model.eval()

    # Test 1: Model randomization
    model_randomization_test(
        model, model_name, sample_img_t, CFG, target_class,
        save_path=f'{sanity_path}/{model_name}_model_rand.png',
        device=DEVICE
    )

    # Test 2: Data randomization (re-trains on shuffled labels)
    data_randomization_test(
        model_name, CFG, test_dataset, target_class,
        save_path=f'{sanity_path}/{model_name}_data_rand.png',
        shuffle_epochs=10,  # increase for more thorough training on shuffled data
        device=DEVICE
    )

print('\nAdebayo sanity checks complete. Check outputs/sanity_checks/')

[sanity] Model randomization grid saved → ./outputs/sanity_checks/baseline_cnn_model_rand.png
[sanity] Training model on shuffled labels (10 epochs)...
  Shuffle epoch 1/10: loss=2.3072 acc=9.7%
  Shuffle epoch 2/10: loss=2.3028 acc=10.0%
  Shuffle epoch 3/10: loss=2.3028 acc=10.1%
  Shuffle epoch 4/10: loss=2.3028 acc=9.9%
  Shuffle epoch 5/10: loss=2.3028 acc=9.8%
  Shuffle epoch 6/10: loss=2.3028 acc=9.8%
  Shuffle epoch 7/10: loss=2.3029 acc=10.0%
  Shuffle epoch 8/10: loss=2.3029 acc=9.9%
  Shuffle epoch 9/10: loss=2.3028 acc=9.9%
  Shuffle epoch 10/10: loss=2.3029 acc=9.6%
[sanity] Data randomization comparison saved → ./outputs/sanity_checks/baseline_cnn_data_rand.png
[sanity] Model randomization grid saved → ./outputs/sanity_checks/resnet18_scratch_model_rand.png
[sanity] Training model on shuffled labels (10 epochs)...
[model] ResNet18 initialized from scratch (random weights)
  Shuffle epoch 1/10: loss=2.3773 acc=10.0%
  Shuffle epoch 2/10: loss=2.3417 acc=9.9%
  Shuffle epoc

---
## 11. Summary & Results
Print a final summary of all results to fill into `LOG.md`.

In [26]:
print('=' * 60)
print('CIFAR-10 Grad-CAM Project — Final Results Summary')
print('=' * 60)

print('\n[ Accuracy Table ]')
print(f'{"Model":<25} {"Test Acc":>10} {"CE Loss":>10}')
print('-' * 47)
for name, res in eval_results.items():
    print(f'{name:<25} {res["accuracy"]:>9.2f}% {res["loss"]:>10.4f}')

print('\n[ Library Parity (Spearman r) ]')
threshold = CFG['evaluation']['library_parity_threshold']
print(f'{"Model":<25} {"r":>8} {"Pass?":>8}')
print('-' * 43)
for name, r in parity_results.items():
    passed = '✓' if (not np.isnan(r) and r >= threshold) else '✗'
    print(f'{name:<25} {r:>8.4f} {passed:>8}')

print('\n[ Output Files ]')
for pattern in [
    'outputs/curves/*_curves.png',
    'outputs/eval/*',
    'outputs/heatmaps/*',
    'outputs/sanity_checks/*',
    'models/*_best.pth',
]:
    import glob
    files = sorted(glob.glob(pattern))
    for f in files:
        print(f'  {f}')

print('\nDone! Fill results into LOG.md.')

CIFAR-10 Grad-CAM Project — Final Results Summary

[ Accuracy Table ]
Model                       Test Acc    CE Loss
-----------------------------------------------
baseline_cnn                  87.28%     0.3825
resnet18_scratch              93.35%     0.2823
resnet18_pretrained           95.86%     0.1640

[ Library Parity (Spearman r) ]
Model                            r    Pass?
-------------------------------------------
baseline_cnn                1.0000        ✓
resnet18_scratch            1.0000        ✓
resnet18_pretrained         1.0000        ✓

[ Output Files ]
  outputs/curves\baseline_cnn_curves.png
  outputs/curves\resnet18_pretrained_curves.png
  outputs/curves\resnet18_scratch_curves.png
  outputs/eval\accuracy_comparison.png
  outputs/eval\baseline_cnn_confusion.png
  outputs/eval\resnet18_pretrained_confusion.png
  outputs/eval\resnet18_scratch_confusion.png
  outputs/heatmaps\baseline_cnn_gradcam_grid.png
  outputs/heatmaps\resnet18_pretrained_gradcam_grid.png
  ou